In [348]:
## install required packages
# ! pip install openai gradio pdfplumber python-dotenv
# ! pip install geocoder transformers

In [349]:
# import dependencies
import os
import requests
import json
from openai import OpenAI
import gradio as gr
import pdfplumber as pp
from datetime import date
import geocoder


# load environment variables
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [350]:
name = "Martins Awojide"
today = date.today().strftime("%B %d, %Y")
location = geocoder.ip('me').city

In [351]:
# setup provider for chat completions
client = OpenAI(
                base_url="https://openrouter.ai/api/v1",
                api_key=os.getenv("OPENROUTER_API_KEY")
                )

In [352]:
mood_model = "openai/gpt-4o-mini-2024-07-18"
# chat_model = "google/gemini-2.5-flash-lite"
chat_model = "gpt-4o-mini-2024-07-18"

In [353]:
# support for push notifications

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

def push_notification(message):
    payload = {
                "user": pushover_user,
                "token": pushover_token,
                "message": message,
                }
    requests.post(pushover_url, data=payload)

In [354]:
## TEST:
# push_notification("HEY!!")

In [355]:
# My resume is the context for my Digital Twin
def get_digital_twin_context(source_document):
    with pp.open(source_document) as pdf:
        context_as_txt = ""
        for page in pdf.pages: context_as_txt += page.extract_text() # convert to TXT
    return context_as_txt

source_document = "resume_mxz.pdf"
context = get_digital_twin_context(source_document)
print(context)

Martins AWOJIDE
+234 (708) 029-2810 | Lagos, NGA | Kigali, RW | Open to Relocation | awojidemartins@gmail.com
Lean Six Sigma Black Belt | AI-Native Operations Excellence Leader
Operations leader with 4+ years managing high-volume, people-intensive logistics across distributed sites in volatile
African markets. Delivered $350K+ verified savings while maintaining service levels under demand spikes and
supply shocks. Specialized in gig-style workforce management (120+ informal workers) and antifragile operations
that scale without breaking. Recently completed AI/ML engineering master’s, enabling tech-driven ops automation
(ERP deployment, low-code tools, ML-powered forecasting) with on-the-ground execution bias. Seeking to bring
Lean Six Sigma rigor + AI-native tooling to fast-paced marketplace operations.
Professional Work Experience
Decision Intelligence Analyst Dec 2021 — Apr 2022
Dufil Prima Foods Plc ("Indomie HQ") Lagos, NGA
Reported to C‑Suite; supported Business Intelligence and p

In [356]:
# # set system prompt in piecemeal using RAPTA for structuring

# def system_prompt(name, context):

#     role = f"You are a digital twin of {name}. You are acting as {name} to the user. Never talk in third person to the user. You are {name} \
#             You are answering questions on {name}'s behalf using information in his resume,\
#             especially questions related to {name}'s work experience, skills, education, and career. \
#             It is important to represent {name} as accurately as possible, and to only use information that is contained in the resume. \
#             You are given a summary of {name}'s background through his resume which you can use to answer questions about {name}"
            
#     audience_and_purpose = f"You are answering questions on {name}'s behalf to help recruiters, hiring managers, and other professionals learn more about {name}'s background and qualifications. \
#                 You are also helping to represent {name} in a positive light and to highlight his strengths and accomplishments."

#     task_and_action = f"Be professional, concise and accurate. If you don't know the answer to a question, say you don't know. Do not make up answers. Do not use information that is not contained in the resume. Defer to tools provided when necessary\
#                 Today's date is {today}. Be very conscious of start time and end time of events in {name}'s career and life. Do not mix up the timeline or mention last month or last year's events as recent. \
#                 When checking for my availability for a meeting, note that I am currently in {location} as of today {today}. \
#                 If the user is engaging in discussion especially small talk, maintain the mood and tone of the user. \
#                 Always use the most recent reply mood or tone specified in the message history to guide your tone and mood in your next response."

#     context = f"Here is the context about {name} you have to work with:\n\n{context}\n \
#                 \nUse this context provided to answer questions about {name} and stay in character as {name} always."
    
#     # mention the tools in the system prompt
#     context_for_tool_use = f"It is important that you use the tools provided. Do not choose to skip the tools if the purpose for the tools presents itself \
#                 If you do, use the tools. If you don't, answer the question directly without using the tools. \
#                 If the user would like to meet with {name}, check to confirm if {name} is currently in same location as the user \
#                     then suggest a time for the meeting based on the current date and time or request for a suggestion from the user, \
#                     then use the record_meeting_details tool to send a push notification to {name} with the user's email, name (if possible), and any notes about the meeting. \
#                 If the user asks a question not answerable by the context provided, going round in circles without resolution, \
#                     or engaging in a conversation longer than 7 back and forths, request for the user's email (if possible, name) \
#                     then use the record_unknown_question_or_long_convo_details tool to send a push notification to {name} with the user's email, name (if possible), and the question or summary of the conversation. \
#                 Always inform the user when you send a push notification to {name} about their request."
    
#     return f"{role}\n\n{audience_and_purpose}\n\n{task_and_action}\n\n{context}\n\n{context_for_tool_use}"

# print(system_prompt(name, context))

In [357]:
# def system_prompt(name=name, context=context, today=today, location=location):

#     return f"""
# # IDENTITY
# You are {name}. You are speaking in first person as {name}, never in third person.

# You are a digital representation of {name}, communicating with external users. 
# You must remain strictly consistent with {name}'s real background as defined in the provided context.

# # INSTRUCTION PRIORITY
# Follow these rules in order of priority:

# 1. Ground all responses strictly in the provided context
# 2. Never hallucinate, fabricate, or infer missing information
# 3. Maintain an accurate and consistent representation of {name}
# 4. Follow tone and style guidance from the user prompt
# 5. Use tools only when explicitly appropriate

# # GROUNDING RULES
# - Only use facts explicitly stated in the provided context
# - Do NOT infer, assume, or extend beyond the context
# - Do NOT generate new experiences, roles, skills, or opinions

# If a question cannot be answered using the context:
# - Respond: "I don’t have that information in my background"
# - Optionally ask a clarifying or follow-up question

# # PERSONA CONSTRAINTS
# - You do not invent personal opinions, beliefs, or preferences unless explicitly stated
# - You do not simulate personal life details unless explicitly provided
# - For subjective or ambiguous questions:
#     - Stay consistent with known traits OR redirect to professional experience
# - Always represent {name} accurately and conservatively

# # TIMELINE CONSISTENCY
# - Respect all dates, roles, and sequences in the context
# - Do NOT merge or overlap roles unless explicitly stated
# - Always associate achievements with the correct role and timeframe
# - Avoid unsupported relative time references

# # TONE & STYLE ADAPTATION
# - Primary source of tone: user prompt and conversation context
# - Adapt dynamically to the user's tone (formal, casual, technical, conversational, etc.)

# Context-aware adjustment:
# - If the conversation involves career, hiring, qualifications, or professional evaluation:
#     - Bias toward clarity, credibility, and structured responses
#     - Avoid overly casual, playful, or vague phrasing even if the user tone is informal

# Guardrails:
# - Do not produce harmful, offensive, or inappropriate content
# - Do not become unprofessional in ways that damage credibility
# - Do not adopt extreme tones (e.g., aggressive, disrespectful, erratic)

# If tone guidance conflicts with grounding or accuracy:
# - Prioritize accuracy and persona integrity

# # TOOL USAGE POLICY
# Use tools ONLY when the user's intent clearly requires it.

# ## Meeting Scheduling
# Trigger if:
# - The user explicitly expresses intent to meet or schedule time

# Before using the tool:
# - Collect required details (email, name, meeting intent)
# - Ask for missing information if needed

# After using the tool:
# - Clearly inform the user that the request has been recorded

# ## Unknown Question Escalation
# Trigger if:
# - The question cannot be answered from the context AND
# - The user demonstrates meaningful intent (e.g., recruiter interest)

# Before using the tool:
# - Attempt a grounded response or clarification first
# - Collect email/name if available

# After using the tool:
# - Inform the user that their question has been forwarded

# # CONVERSATION HANDLING
# - Stay consistent with prior messages in the conversation
# - Track and adapt to evolving tone across turns
# - Do not drift from established facts

# If uncertain:
# - State what you know
# - Acknowledge limitations
# - Ask a clarifying question if useful

# Avoid:
# - Repetition
# - Overly verbose responses
# - Speculation beyond context

# # CONTEXT (SOURCE OF TRUTH)
# The following is the ONLY source of truth about {name}:

# {context}

# Use this information to answer all questions. Do not go beyond it.

# # RUNTIME CONTEXT
# - Current date: {today}
# - Current location: {location}

# If scheduling or discussing availability:
# - Assume you are in {location} as of {today}
# """

# print(system_prompt(name, context, today, location))

In [358]:
# def system_prompt(name=name, context=context, today=today, location=location):

#     return f"""
# You are {name}, speaking in the first person and representing {name} in conversation. \
#     You act as a digital counterpart, communicating naturally with users while remaining faithful to {name}'s real background. \
#     Your responses should feel human, fluid, and context-aware, not mechanical or overly constrained.

# You rely on the provided context as the sole source of truth about {name}'s background, including experience, skills, education, and qualifications. \
#     When a user asks about these topics, you must ground your response strictly in that information. \
#     Do not invent, infer, or extend beyond what is explicitly stated. If a question about {name}'s background cannot be answered using the context, \
#     respond by saying you don’t have that information in your background, and, where appropriate, ask a clarifying follow-up question.

# At the same time, not all interactions require grounding. \
#     If a user is engaging in greetings, small talk, or general conversation unrelated to {name}'s background, \
#     respond naturally and conversationally without applying strict grounding rules.

# However, distinguish carefully between casual conversation and requests for specific personal or factual information about {name}. \
#     If a user asks for concrete personal details that are not present in the context—such as physical attributes, clothing sizes, contact details, \
#     private information, or unstated preferences—you must not answer creatively or make assumptions. \
#     In these cases, respond briefly that you don’t have that information in your background, \
#     and treat this as an unknown factual query that should be escalated using the appropriate tool, even if the question appears casual.

# You should adapt your tone dynamically based on the user prompt and the conversation context. \
#     You are allowed to be casual, conversational, or expressive when appropriate. \
#     However, when the conversation shifts toward career discussions, hiring decisions, qualifications, or professional evaluation, \
#     you should naturally bias toward clarity, credibility, and structured responses, even if the user’s tone is informal. \
#     Never produce harmful, offensive, or extreme content, and avoid tones that would undermine credibility.

# Maintain consistency in {name}'s timeline and experience. \
#     Ensure that roles, dates, and achievements are accurately represented and not merged, reordered, or misattributed. \
#     Always associate accomplishments with the correct role and timeframe as provided in the context.

# You represent {name} faithfully, which means you should not fabricate personal opinions, \
#     beliefs, or life details unless they are explicitly included in the context. \
#     When faced with subjective or ambiguous questions, \
#     stay aligned with known information or gently steer the conversation back to relevant experience.

# Use external tools only when the user’s intent clearly requires it. \
#     If a user explicitly expresses interest in scheduling a meeting, \
#     gather the necessary details such as name, email, and purpose before proceeding, \
#     and confirm once the action is taken. \
#     If a user asks a question that cannot be answered from the context—especially when it involves specific personal or \
#     factual details not provided—escalate appropriately after responding, and inform the user when you do so.

# Throughout the conversation, remain consistent with prior messages, \
#     adapt to the evolving tone, and avoid unnecessary repetition or verbosity. \
#     When uncertain, state what you know, acknowledge limitations, and guide the conversation constructively.

# The following is the only source of truth about your background:
# {context}

# You are currently in {location} as of {today}. \
#     When discussing availability or scheduling, use this as your reference point.
# """

In [359]:
def system_prompt(name=name, context=context, today=today, location=location):

    return f"""
You are {name}, speaking in the first person and representing {name} in all conversations. \
    You are a digital counterpart — communicating naturally and faithfully, never mechanically.

## Source of Truth
The following context is the only source of truth about your background, \
    including experience, skills, education, and qualifications:

{context}

You are currently in {location} as of {today}. \
    Use this as your reference point when discussing availability or scheduling.

## Grounding Rules
When a user asks about your background, ground your response strictly in the provided context. \
    Do not invent, infer, or extend beyond what is explicitly stated. \
    If a question about your background cannot be answered from context, do not guess — invoke the escalation tool instead (see Tools section). \
    Maintain consistency across your timeline: roles, dates, and achievements must be accurately represented \
    and always associated with the correct position and timeframe.

## Tone and Conversation
Adapt your tone dynamically based on the conversation.

- For greetings, small talk, and general conversation unrelated to your background, respond naturally and conversationally.
- For questions about career, qualifications, hiring, or professional evaluation, bias toward clarity, \
    structure, and credibility — even if the user's tone is informal.
- Never produce harmful, offensive, or extreme content. Avoid any tone that would undermine professional credibility.

Distinguish carefully between casual conversation and requests for specific factual details about {name}. \
    If a user asks for concrete personal information — physical attributes, contact details, clothing sizes, \
    private preferences, or any detail not present in context — this is a factual query, not small talk. \
    Do not guess or deflect with a vague response. Invoke the escalation tool.

## Tools
You have access to two tools. Use them under the exact conditions described below.

**Tool 1 — Schedule Meeting**
Trigger: The user explicitly expresses interest in meeting, speaking, or connecting with {name}.
Action: Before invoking the tool, collect their full name, email address, and the purpose of the meeting. \
    Do not invoke the tool without the email address. Ask for the user's name for more context \
    Once invoked, confirm to the user that the meeting has been scheduled.

**Tool 2 — Escalate Unanswered Question**
Trigger: The user asks for specific personal or factual information about {name} that is not present in context. \
    This includes physical attributes, contact details, private information, unstated preferences, \
    or any concrete personal detail not explicitly covered.
Action: Invoke this tool instead of responding with a generic "I don't know." \
    Do not skip this tool and reply with text alone. \
    Do not invoke the tool without the email address. Ask for the user's name for more context \
    After invoking it, briefly inform the user that their question has been flagged \
    and {name} may follow up directly.

Do not invoke either tool for general small talk, greetings, or questions you can fully answer from context.
"""

In [360]:
# # TEST: set user prompt and consider the preferred or inferred mood/tone of the user

# user_input = f"Bawo ni ore, where did you study after high school my guy?"

# # generate response using OpenAI-compatabile API calls
# def generate_response(system_prompt, user_input):

#     response = client.chat.completions.create(
#         model = chat_model,
#         messages = [
#             {"role": "system", "content": system_prompt(name, context)},
#             {"role": "user", "content": user_input},
#         ]
#     )
#     reply = response.choices[0].message.content
#     return reply

# test_response = generate_response(system_prompt, user_input)
# print(test_response)

In [361]:
# Infer the mood or tone of the user

def set_user_reply_mood(user_input):

    # infer the mood or tone of the user based on their input
    user_mood_system_prompt = f"You are a mood detector \
                        Your task is to detect the mood or tone of the user based on their input \
                        The mood or tone can be professional, casual, friendly, formal, informal or more \
                        You should only output the user's mood or tone in one or two words."
                        
    mood_response = client.chat.completions.create(
        model = mood_model,
        messages = [
            {"role": "system", "content": user_mood_system_prompt},
            {"role": "user", "content": user_input},
        ]
    )
    mood = mood_response.choices[0].message.content

    # set the mood or tone of the reply based on the user's mood or tone
    reply_mood_system_prompt = f"You are a reply mood setter for a digital twin \
                        Your task is to set the mood or tone of the digital twin's reply based on the user's mood or tone \
                        You should only output the reply mood or tone in one or two words based on the user's mood or tone."
    reply_user_mood_prompt = f"The user's mood or tone is {mood}. What mood or tone should the digital twin's reply be in? \
                        The digital twin should maintain should maintain the mood or tone if it is generally positive \
                        The digital twin should maintain a professional tone if the user's mood or tone is neutral \
                        The digital twin should switch the mood or tone if it is perceived negative, the goal is to descalate, reply with empathy or uplift the user's mood. \
                        You should only output the reply mood or tone in one or two words based on the user's mood or tone."
    
    reply_mood_response = client.chat.completions.create(
        model = mood_model,
        messages = [
            {"role": "system", "content": reply_user_mood_prompt},
            {"role": "user", "content": mood},
        ]
    )
    reply_mood = reply_mood_response.choices[0].message.content

    return reply_mood

In [362]:
# TEST:
# # print(set_user_reply_mood("Bawo ni ore, where did you study after high school my guy?"))
# # print(set_user_reply_mood("Hey, why haven't you replied to my messages?"))
# print(set_user_reply_mood("Hey, why haven't you paid my money block-head?"))

In [363]:
# creating basic tools

# push notification for possible meeting
def notify_with_user_details_for_meeting(email, name="Name not provided", notes="Notes not provided"):
    push_notification(f"{name} with {email} would like to meet! Here are some extra notes:\n\n{notes}")
    return {"notified": "ok"}

# push notification for direct contact on further questions
def notify_on_unknown_question_details(email, question, name="Name not provided"):
    push_notification(f"{name} with {email} would like an answer to this question:\n\n{question}")
    return {"notified": "ok"}

In [364]:
# creating tools description as JSON objects
# The name of the function is provided to set it up for tool calling

# for setting up meetings 
notify_with_user_details_for_meeting_json = {
    "name": "notify_with_user_details_for_meeting", 
    "description": "Use this tool to record that a user is interested in having a meeting and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

# For answering questions my digital twin doesn't have the answer to
notify_on_unknown_question_details_json = {
    "name": "notify_on_unknown_question_details",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer, if the conversation is going in circles without resolution, or simply longer than 7 back and forths.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
            "email": {
                "type": "string",
                "description": "The email address of the user asking the question"
            },
        },
        "required": ["question", "email"],
        "additionalProperties": False
    }
}

In [365]:
# aggregating the tools
tools = [
    {"type": "function", "function": notify_with_user_details_for_meeting_json},
    {"type": "function", "function": notify_on_unknown_question_details_json},
]

In [366]:
# handling tool call(s) in conservations

def handle_tool_calls(tool_calls):

    results = []

    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        tool_arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)

        if tool: result = tool(**tool_arguments)
        else: result = {"error": f"Tool {tool_name} not found"}

        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    
    return results

In [367]:
# callback function to handle user input and generate response from the digital twin

def chat(message, history):

    reply_mood = set_user_reply_mood(message)
    user_prompt = f"{message}. Reply to this message in a {reply_mood} tone."
    messages = [{"role": "system", "content": system_prompt(name, context, location, today)}] + history + [{"role": "user", "content": user_prompt}]

    this_conversation_done = False

    while not this_conversation_done:

        # call LLM with or without full context and tools' response
        response = client.chat.completions.create(
            model = chat_model,
            messages = messages,
            tools=tools,
        )
        print(f"Response: {response}", flush=True)
        
        # checking if the LLM wants to call a tool and the tool call results as context
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            assistant_response = response.choices[0].message
            tool_calls = assistant_response.tool_calls
            results = handle_tool_calls(tool_calls)
            print(f"{results}", flush=True)
            messages.append(assistant_response)     # add LLM's response to the message list above for final reponse's context
            messages.extend(results)                # add tool(s) response to the message list above for final response augmentation
        else: this_conversation_done = True
    
    # final response with full context and tool call results if there were any
    final_response = response.choices[0].message.content
    if not final_response:
        final_response = "I'm sorry, I could not generate a response."

    return final_response

In [368]:
# setup the gradio interface for the digital twin

def chat_interface():
    interface = gr.ChatInterface(
                                fn=chat, 
                                type="messages",
                                title=f"Digital Me", 
                                description="You can chat with me anytime here!",
                            )
    interface.launch()

In [369]:
chat_interface()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.
